In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Deploy your own System One model on Gemini platform

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/open-models/serving/gemini_platform_vllm_diffusiongemma_structured_outputs.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fopen-models%2Fserving%2Fgemini_platform_vllm_diffusiongemma_structured_outputs.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/open-models/serving/gemini_platform_vllm_diffusiongemma_structured_outputs.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/open-models/serving/gemini_platform_vllm_diffusiongemma_structured_outputs.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/open-models/serving/gemini_platform_vllm_diffusiongemma_structured_outputs.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/open-models/serving/gemini_platform_vllm_diffusiongemma_structured_outputs.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/open-models/serving/gemini_platform_vllm_diffusiongemma_structured_outputs.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/open-models/serving/gemini_platform_vllm_diffusiongemma_structured_outputs.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/open-models/serving/gemini_platform_vllm_diffusiongemma_structured_outputs.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| Author(s) |
| --- |
| [Ariel Jassan](https://github.com/arieljassan) |

## Overview

In this notebook, we will deploy the [**DiffusionGemma 26B-A4B**](https://huggingface.co/nvidia/diffusiongemma-26B-A4B-it-NVFP4) model into an Endpoint for online predictions on the Gemini platform, giving you a **System One** model capable of serving low-latency structured decision requests.

DiffusionGemma 26B-A4B is Google DeepMind's discrete block-diffusion language model adapted from Gemma 4 (`26B` total parameters, `4B` active per token). Unlike autoregressive models that decode one token at a time from left to right, DiffusionGemma uses **bidirectional attention** across a fixed-width token canvas (`64` tokens per block).

Using official upstream [**vLLM (`vllm/vllm-openai:v0.31.0`)**](https://hub.docker.com/r/vllm/vllm-openai), you can serve the NVFP4-quantized [`nvidia/diffusiongemma-26B-A4B-it-NVFP4`](https://huggingface.co/nvidia/diffusiongemma-26B-A4B-it-NVFP4) checkpoint on a single 24 GB **`NVIDIA_L4`** GPU (`g2-standard-8`) and run both:

1. **Single-Step Structured Diffusion Reads (System One)**: By seeding a partially masked JSON canvas (`diffusion_seed_canvas`, `diffusion_read_only: True`, `diffusion_max_steps: 1`, `diffusion_constrained: True`), vLLM evaluates all decision fields simultaneously in a **single forward pass** and returns calibrated marginal probabilities and Shannon entropy confidence scores:
   - **`choice`**: Categorical selection (`A`–`Z`) mapped back to your choice keys.
   - **`score`**: Ordinal rating (`0`–`9`) with expected value (`sum(i * p_i)`).
   - **`boolean`**: Binary/epistemic truth probability (`P(true) / (P(true) + P(false))`).
2. **Standard Block-Diffusion Chat Generation**: Multi-step parallel block decoding via standard `/v1/chat/completions`.

### What you'll cover

- Create an Artifact Registry remote repository for Docker Hub to serve the official upstream vLLM container (`docker.io/vllm/vllm-openai:v0.31.0`).
- Deploy `nvidia/diffusiongemma-26B-A4B-it-NVFP4` to an Endpoint for online predictions on the Gemini platform (`g2-standard-8` with `1x NVIDIA_L4`).
- Compile a multi-question structured decision schema into a masked token canvas using the official vLLM structured diffusion protocol.
- Send single-step structured diffusion requests and standard block-diffusion chat requests using `endpoint.raw_predict`.
- Clean up the deployed endpoint and model resources.

## Get started

### Install required packages

In [ ]:
%pip install --upgrade --quiet google-cloud-aiplatform transformers

### Authenticate your notebook environment (Colab only)

If you're running this notebook on Google Colab, run the following cell to authenticate your environment.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Set Google Cloud project information and initialize the SDK

To get started using the Gemini platform, you must have an existing Google Cloud project and [enable the API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [ ]:
import json
import math
import time
from typing import Any

import google.auth
from google.cloud import aiplatform
from transformers import AutoTokenizer

# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
LOCATION = "us-central1"  # @param {type: "string"}
ENDPOINT_NAME = "diffusiongemma-vllm"  # @param {type: "string"}
# fmt: on
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    _, adc_project = google.auth.default()
    PROJECT_ID = str(adc_project)

SERVED_MODEL_NAME = "diffusiongemma-26b"
HF_MODEL_ID = "nvidia/diffusiongemma-26B-A4B-it-NVFP4"

aiplatform.init(project=PROJECT_ID, location=LOCATION)
print(f"Initialized Gemini platform for project={PROJECT_ID} in {LOCATION}")

## Create an Artifact Registry Remote Repository for Docker Hub

Online Prediction on the Gemini platform requires serving containers to be served from Google Cloud Artifact Registry. Run the following cell once in your project to create an [Artifact Registry remote repository](https://cloud.google.com/artifact-registry/docs/repositories/remote-repo) for Docker Hub so the Gemini platform can pull and cache [`vllm/vllm-openai:v0.31.0`](https://hub.docker.com/r/vllm/vllm-openai) directly on demand:

In [ ]:
!gcloud artifacts repositories create docker-hub \
    --repository-format=docker \
    --mode=remote-repository \
    --remote-docker-repo=DOCKER_HUB \
    --location=us \
    --project={PROJECT_ID}

## Deploy DiffusionGemma 26B-A4B to a dedicated Endpoint on the Gemini platform

Next, register the official vLLM serving container in Model Registry and deploy it to a Dedicated Endpoint on the Gemini platform using a `g2-standard-8` (`1x NVIDIA_L4`, 24 GB VRAM) instance.

We configure vLLM with:
- `--diffusion-config={"canvas_length": 64}` for 64-token block diffusion.
- `--attention-backend=TRITON_ATTN` and `--limit-mm-per-prompt={"image": 0, "video": 0}` so the NVFP4-quantized 26B-A4B model fits comfortably in 24 GB VRAM.
- `--enable-prefix-caching` and `--max-logprobs=32` for fast structured diffusion reads.

In [ ]:
# Official upstream vLLM release image served via Artifact Registry remote repo.
VLLM_IMAGE = (
    f"us-docker.pkg.dev/{PROJECT_ID}/docker-hub/vllm/vllm-openai:v0.31.0"
)

diffusion_cfg = json.dumps({"canvas_length": 64})
mm_limit_cfg = json.dumps({"image": 0, "video": 0})
gen_override_cfg = json.dumps({"max_new_tokens": None})

container_args = [
    # Model weights on Hugging Face and short OpenAI API model alias.
    HF_MODEL_ID,
    f"--served-model-name={SERVED_MODEL_NAME}",
    # Bind address and port expected by Gemini platform Online Prediction.
    "--host=0.0.0.0",
    "--port=8080",
    # Block-diffusion width and max top_logprobs per token.
    f"--diffusion-config={diffusion_cfg}",
    "--max-logprobs=32",
    # Clear default max_new_tokens so per-request max_tokens takes effect.
    f"--override-generation-config={gen_override_cfg}",
    # Gemma 4 chat-template parsers for reasoning and tool calls.
    "--enable-auto-tool-choice",
    "--tool-call-parser=gemma4",
    "--reasoning-parser=gemma4",
    # Latency optimizations (KV prefix cache + overlapped CPU/GPU steps).
    "--enable-prefix-caching",
    "--async-scheduling",
    # Fit 17 GB NVFP4 weights + KV cache inside 1x 24 GB NVIDIA_L4 GPU.
    # Disable multimodal vision/video encoders for text-only.
    f"--limit-mm-per-prompt={mm_limit_cfg}",
    # Use Triton attention backend for bidirectional diffusion masks.
    "--attention-backend=TRITON_ATTN",
    "--max-num-seqs=1",
    "--max-model-len=4096",
    "--gpu-memory-utilization=0.92",
]

container_env = {"VLLM_USE_V2_MODEL_RUNNER": "1"}

model = aiplatform.Model.upload(
    display_name=ENDPOINT_NAME,
    model_id=ENDPOINT_NAME,
    serving_container_image_uri=VLLM_IMAGE,
    serving_container_command=["vllm", "serve"],
    serving_container_args=container_args,
    serving_container_ports=[8080],
    serving_container_predict_route="/v1/chat/completions",
    serving_container_health_route="/health",
    serving_container_environment_variables=container_env,
)

endpoint = aiplatform.Endpoint.create(
    display_name=ENDPOINT_NAME,
    endpoint_id=ENDPOINT_NAME,
    dedicated_endpoint_enabled=True,
)

endpoint.deploy(
    model=model,
    deployed_model_display_name=ENDPOINT_NAME,
    machine_type="g2-standard-8",
    accelerator_type="NVIDIA_L4",
    accelerator_count=1,
    min_replica_count=1,
    max_replica_count=1,
)

print(f"Endpoint resource: {endpoint.resource_name}")
print(f"Dedicated DNS:     {endpoint.dedicated_endpoint_dns}")

## Compile a structured decision rubric for single-step diffusion

In vLLM `v0.31.0+`, DiffusionGemma supports **single-step structured diffusion** via `vllm_xargs` on `/v1/chat/completions` ([vllm-project/vllm#57250](https://github.com/vllm-project/vllm/pull/57250), [#58216](https://github.com/vllm-project/vllm/pull/58216)).

Instead of generating JSON tokens one by one, the client compiles a structured schema into a **seeded token canvas** (`diffusion_seed_canvas`) where structural JSON tokens (`{"q0": "`, `", "q1": `, etc.) are already known and only the answer slots contain `<|Reserved_2|>` (`token_id=69`) mask tokens:

- `diffusion_max_steps: 1`: Runs a single bidirectional forward pass across the canvas.
- `diffusion_read_only: True`: Reads the marginal log-probabilities at every masked slot without mutating the canvas.
- `diffusion_constrained: True`: Restricts vocabulary logits to `logprob_token_ids`.
- `diffusion_canvas_length`: Narrowed to the smallest multiple of 16 that fits the JSON template (e.g., `32` tokens for 4 questions).

In [ ]:
# System instruction priming the model to output raw JSON without prose.
SYSTEM_PROMPT = (
    "You are a structured assessment engine. Read the state carefully and"
    " answer every question simultaneously. Return ONLY the JSON object."
)


class StructuredDiffusionCompiler:
    """Compiles structured schemas into vLLM DiffusionGemma canvas requests."""

    def __init__(self, tokenizer_id: str = HF_MODEL_ID) -> None:
        self.tok = AutoTokenizer.from_pretrained(tokenizer_id)
        # Token ID 69 (<|Reserved_2|>) is DiffusionGemma's [MASK] placeholder.
        self.mask_id: int = self.tok.convert_tokens_to_ids("<|Reserved_2|>")
        self.pad_id: int = self.tok.pad_token_id or 0
        # Precompute single-token IDs for letters (A-Z), digits (0-9), and
        # booleans (true/false) so every answer fits in a 1-token mask slot.
        self.letter_ids: list[int] = [
            self._single_token_id(chr(ord("A") + i)) for i in range(26)
        ]
        self.digit_ids: list[int] = [
            self._single_token_id(str(d)) for d in range(10)
        ]
        self.true_id: int = self._single_token_id("true")
        self.false_id: int = self._single_token_id("false")

    def _single_token_id(self, text: str) -> int:
        ids = self.tok.encode(text, add_special_tokens=False)
        return ids[0]

    def compile_schema(self, questions: dict[str, dict[str, Any]]) -> dict:
        # Build the user prompt rubric and the seeded JSON token canvas.
        prompt_parts: list[str] = ["### Questions\n"]
        canvas_Ids: list[int] = []
        slots: list[dict[str, Any]] = []
        all_label_ids: set[int] = set()

        def emit(text: str) -> None:
            canvas_Ids.extend(self.tok.encode(text, add_special_tokens=False))

        # Start the JSON object on the canvas.
        emit("{")
        items = list(questions.items())
        for idx, (q_name, q_spec) in enumerate(items):
            qid = f"q{idx}"
            q_type = q_spec["type"]
            instr = q_spec["instructions"]

            if q_type == "choice":
                # Map multi-token choice keys to single-token letters (A-Z).
                criteria = q_spec["criteria"]
                keys = list(criteria.keys())
                letters = [chr(ord("A") + j) for j in range(len(keys))]
                label_ids = self.letter_ids[: len(keys)]
                lines = [
                    f"  {letters[j]} = {keys[j]}: {criteria[keys[j]]}"
                    for j in range(len(keys))
                ]
                prompt_parts.append(
                    f"[{qid}] ({q_name}) {instr}\n"
                    f"Answer with a single letter ({'/'.join(letters)}):\n"
                    + "\n".join(lines)
                    + "\n"
                )
                # Append '"q0": "[MASK]"' to the canvas and record the index.
                emit(f'"{qid}": "')
                pos = len(canvas_Ids)
                canvas_Ids.append(self.mask_id)
                emit('"' + (", " if idx + 1 < len(items) else ""))
                slots.append({
                    "name": q_name,
                    "type": "choice",
                    "pos": pos,
                    "label_ids": label_ids,
                    "keys": keys,
                })
                all_label_ids.update(label_ids)
            elif q_type == "score":
                # Map ordinal rubric levels to single-digit tokens (0..N-1).
                criteria_list = q_spec["criteria"]
                n_levels = len(criteria_list)
                label_ids = self.digit_ids[:n_levels]
                legend = {str(j): criteria_list[j] for j in range(n_levels)}
                lines = [
                    f"  {j} = {criteria_list[j]}" for j in range(n_levels)
                ]
                prompt_parts.append(
                    f"[{qid}] ({q_name}) {instr}\n"
                    f"Answer with a single digit (0-{n_levels - 1}):\n"
                    + "\n".join(lines)
                    + "\n"
                )
                # Append '"q1": [MASK]' to the canvas and record the index.
                emit(f'"{qid}": ')
                pos = len(canvas_Ids)
                canvas_Ids.append(self.mask_id)
                if idx + 1 < len(items):
                    emit(", ")
                slots.append({
                    "name": q_name,
                    "type": "score",
                    "pos": pos,
                    "label_ids": label_ids,
                    "legend": legend,
                })
                all_label_ids.update(label_ids)
            elif q_type == "boolean":
                # Map binary questions to single-token 'true' and 'false' IDs.
                label_ids = [self.true_id, self.false_id]
                prompt_parts.append(
                    f"[{qid}] ({q_name}) {instr}\nAnswer true or false.\n"
                )
                emit(f'"{qid}": ')
                pos = len(canvas_Ids)
                canvas_Ids.append(self.mask_id)
                if idx + 1 < len(items):
                    emit(", ")
                slots.append({
                    "name": q_name,
                    "type": "boolean",
                    "pos": pos,
                    "label_ids": label_ids,
                })
                all_label_ids.update(label_ids)

        # Close the JSON object and pad the canvas to a multiple of 16 tokens.
        emit("}")
        template_len = len(canvas_Ids)
        canvas_width = max(16, ((template_len + 15) // 16) * 16)
        seed_canvas = canvas_Ids + [self.pad_id] * (canvas_width - template_len)
        return {
            "question_block": "\n".join(prompt_parts),
            "seed_canvas": seed_canvas,
            "canvas_width": canvas_width,
            "slots": slots,
            "all_label_ids": sorted(all_label_ids),
        }

    def build_request(
        self, state: str, compiled: dict[str, Any], model: str
    ) -> dict[str, Any]:
        # Combine the input state and compiled rubric into a user prompt.
        user_msg = (
            f"### State\n{state}\n\n{compiled['question_block']}\n"
            "Fill in every field of the JSON object accurately."
        )
        return {
            "model": model,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg},
            ],
            "max_tokens": compiled["canvas_width"],
            "logprobs": True,
            "top_logprobs": len(compiled["all_label_ids"]),
            # Restrict logprobs to valid option IDs and return raw token IDs.
            "logprob_token_ids": compiled["all_label_ids"],
            "return_tokens_as_token_ids": True,
            "chat_template_kwargs": {"enable_thinking": False},
            # Configure 1-step read-only constrained diffusion over the canvas.
            "vllm_xargs": {
                "diffusion_seed_canvas": compiled["seed_canvas"],
                "diffusion_max_steps": 1,
                "diffusion_read_only": True,
                "diffusion_constrained": True,
                "diffusion_canvas_length": compiled["canvas_width"],
            },
        }

    def parse_response(
        self, raw_json: dict[str, Any], compiled: dict[str, Any]
    ) -> dict[str, Any]:
        # Extract log-probabilities at each mask slot and normalize answers.
        content = raw_json["choices"][0]["logprobs"]["content"]
        answers: dict[str, Any] = {}
        for slot in compiled["slots"]:
            entry = content[slot["pos"]]
            lp_map: dict[int, float] = {}
            # Parse token IDs from vLLM's 'token_id:<id>' string format.
            for cand in entry.get("top_logprobs", []):
                tid = int(cand["token"].split(":")[-1])
                lp_map[tid] = float(cand["logprob"])

            # Softmax over the valid label token IDs for this question.
            raw_logprobs = [lp_map.get(t, -100.0) for t in slot["label_ids"]]
            max_lp = max(raw_logprobs)
            exp_vals = [math.exp(lp - max_lp) for lp in raw_logprobs]
            total = sum(exp_vals) or 1.0
            probs = [e / total for e in exp_vals]

            # Compute normalized Shannon entropy confidence: 1 - H(p) / ln(K).
            k = len(probs)
            entropy = -sum(p * math.log(p) for p in probs if p > 0)
            conf = 1.0 - (entropy / math.log(k)) if k > 1 else 1.0

            if slot["type"] == "choice":
                # Map single-letter probabilities back to original choice keys.
                keys = slot["keys"]
                prob_dict = {keys[i]: round(probs[i], 6) for i in range(k)}
                best_idx = max(range(k), key=lambda i: probs[i])
                answers[slot["name"]] = {
                    "type": "choice",
                    "choice": keys[best_idx],
                    "probabilities": prob_dict,
                    "confidence": round(conf, 6),
                }
            elif slot["type"] == "score":
                # Compute the probability-weighted expected score: sum(i * p_i).
                prob_dict = {str(i): round(probs[i], 6) for i in range(k)}
                expected = sum(i * probs[i] for i in range(k))
                answers[slot["name"]] = {
                    "type": "score",
                    "score": round(expected, 6),
                    "legend": slot["legend"],
                    "probabilities": prob_dict,
                    "confidence": round(conf, 6),
                }
            elif slot["type"] == "boolean":
                # Return P(true) for binary boolean questions.
                answers[slot["name"]] = {
                    "type": "boolean",
                    "boolean": round(probs[0], 6),
                }

        usage = raw_json.get("usage", {})
        return {
            "model": raw_json.get("model", SERVED_MODEL_NAME),
            "answers": answers,
            "usage": {
                "input_tokens": usage.get("prompt_tokens", 0),
                "output_tokens": 0,
            },
        }


compiler = StructuredDiffusionCompiler(HF_MODEL_ID)
print("Initialized StructuredDiffusionCompiler.")

## Send a single-step structured diffusion request

Use `endpoint.raw_predict(use_dedicated_endpoint=True)` to evaluate an enterprise support ticket across four structured questions (`routing_team`, `severity_level`, `churn_or_expansion_risk`, and `requires_immediate_page`) simultaneously in a single forward pass:

In [ ]:
example_state = (
    "customer_tier: Enterprise Plus\n"
    "subject: URGENT: Production SSO login failing after cert rotation\n"
    "body: Hi team, after rotating our SAML certificate 20 minutes ago,"
    " none of our 400 engineers can log into the dashboard. We have a"
    " release freeze in 2 hours and if we cannot resolve this today we"
    " will need to pause our upcoming Q4 contract expansion."
)

example_questions = {
    "routing_team": {
        "type": "choice",
        "instructions": (
            "Which specialized engineering team should own this ticket?"
        ),
        "criteria": {
            "identity_and_security": (
                "SSO, SAML, OAuth, certificates, authentication, login issues"
            ),
            "billing_and_accounts": (
                "invoices, refunds, payment methods, subscription charges"
            ),
            "core_platform_infra": (
                "database outages, storage corruption, compute provisioning"
            ),
        },
    },
    "severity_level": {
        "type": "score",
        "instructions": "What is the operational severity of this incident?",
        "criteria": [
            "low - general question or minor cosmetic issue",
            "medium - partial feature issue with a workaround",
            "critical - production blocker preventing users from working",
        ],
    },
    "churn_or_expansion_risk": {
        "type": "boolean",
        "instructions": (
            "Is the customer signaling risk to a contract renewal,"
            " expansion, or threatening to cancel?"
        ),
    },
    "requires_immediate_page": {
        "type": "boolean",
        "instructions": (
            "Does this issue block production access and require paging the"
            " on-call engineer immediately?"
        ),
    },
}

compiled_schema = compiler.compile_schema(example_questions)
request_payload = compiler.build_request(
    example_state, compiled_schema, model=SERVED_MODEL_NAME
)

t0 = time.perf_counter()
response = endpoint.raw_predict(
    body=json.dumps(request_payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    use_dedicated_endpoint=True,
)
elapsed_ms = (time.perf_counter() - t0) * 1000.0
response.raise_for_status()

parsed = compiler.parse_response(response.json(), compiled_schema)
print(f"HTTP {response.status_code} | Client RTT: {elapsed_ms:.1f} ms")
print(json.dumps(parsed, indent=2))

## Send a block-diffusion chat completion request

Because the endpoint runs the official vLLM OpenAI-compatible server, you can also send standard `/v1/chat/completions` requests for free-form block-diffusion text generation:

In [ ]:
chat_payload = {
    "model": SERVED_MODEL_NAME,
    "messages": [
        {
            "role": "user",
            "content": (
                "In two short sentences, explain how discrete diffusion"
                " language models generate tokens."
            ),
        }
    ],
    "max_tokens": 128,
}

t0 = time.perf_counter()
chat_resp = endpoint.raw_predict(
    body=json.dumps(chat_payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    use_dedicated_endpoint=True,
)
chat_ms = (time.perf_counter() - t0) * 1000.0
chat_resp.raise_for_status()

reply = chat_resp.json()["choices"][0]["message"]["content"]
print(f"HTTP {chat_resp.status_code} | Client RTT: {chat_ms:.1f} ms\n")
print(reply)

## Cleaning up

To avoid incurring ongoing GPU charges, undeploy and delete the Endpoint and Model resources on the Gemini platform when you are finished:

In [ ]:
endpoint.delete(force=True)
model.delete()